In [115]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from mlxtend.frequent_patterns import apriori, association_rules
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn import linear_model
import math
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
path = "Speed Dating Data.csv"
df_raw = pd.read_csv(path, encoding="latin1")
df_raw.shape

(8378, 195)

In [ ]:
df1 = df_raw.copy()
df1 = df1.iloc[:,:98]

df1 = df1.drop(columns = ['id','idg','condtn','position','positin1','order','partner'])
# Danh sách các cột missing data

In [244]:
# 3 nhóm thuộc tính missing lần luợt: <10%, 10-50%, >50%
df2 = df1.drop(columns = ['wave', 'round', 'pid', 'match', 'int_corr','dec','dec_o', 'age_o', 'race_o', 'pf_o_att', 'pf_o_sin',
       'pf_o_int', 'pf_o_fun', 'pf_o_amb', 'pf_o_sha', 'attr_o', 'sinc_o',
       'intel_o', 'fun_o', 'amb_o', 'shar_o', 'like_o', 'prob_o', 'met_o','samerace'])
df2 = df2.drop_duplicates()
miss10 = []
miss50 = []
miss100 = [] 
for v in df2:
    percents = df2[v].isnull().sum()/len(df2)
    if (percents < 0.1) & (percents > 0):
        miss10.append(v)
    elif (percents >= 0.1) & (percents <= 0.5):
        miss50.append(v)
    elif percents > 0.5: miss100.append(v)
print('Số thuộc tính miss < 10%:',len(miss10),'\nSố thuộc tính miss >10-50%:', len(miss50),'\nSố thuộc tính miss > 50%:', len(miss100))

print(miss50)

Số thuộc tính miss < 10%: 47 
Số thuộc tính miss >10-50%: 14 
Số thuộc tính miss > 50%: 3
['undergra', 'zipcode', 'income', 'attr4_1', 'sinc4_1', 'intel4_1', 'fun4_1', 'amb4_1', 'shar4_1', 'attr5_1', 'sinc5_1', 'intel5_1', 'fun5_1', 'amb5_1']


In [243]:
#Loại bỏ các thuộc tính miss trên 50%
df3 = df2.drop(columns = miss100)
#Sử lý nhóm missing < 10%
print(df3.columns)

Index(['iid', 'gender', 'age', 'field', 'field_cd', 'undergra', 'race',
       'imprace', 'imprelig', 'from', 'zipcode', 'income', 'goal', 'date',
       'go_out', 'career', 'career_c', 'sports', 'tvsports', 'exercise',
       'dining', 'museums', 'art', 'hiking', 'gaming', 'clubbing', 'reading',
       'tv', 'theater', 'movies', 'concerts', 'music', 'shopping', 'yoga',
       'exphappy', 'attr1_1', 'sinc1_1', 'intel1_1', 'fun1_1', 'amb1_1',
       'shar1_1', 'attr4_1', 'sinc4_1', 'intel4_1', 'fun4_1', 'amb4_1',
       'shar4_1', 'attr2_1', 'sinc2_1', 'intel2_1', 'fun2_1', 'amb2_1',
       'shar2_1', 'attr3_1', 'sinc3_1', 'fun3_1', 'intel3_1', 'amb3_1',
       'attr5_1', 'sinc5_1', 'intel5_1', 'fun5_1', 'amb5_1'],
      dtype='str')


In [201]:
#xử lý int_corr
ic_data = df2[['iid','pid','sports', 'tvsports', 'exercise', 'dining', 'museums', 
              'art', 'hiking', 'gaming', 'clubbing', 'reading', 
              'tv', 'theater', 'movies', 'concerts', 'music', 
              'shopping', 'yoga','int_corr']]

person = ic_data.drop(columns = ['pid','int_corr'])
person = person.drop_duplicates()
df_merged = ic_data.merge(
    person,
    left_on='pid', 
    right_on='iid', 
    suffixes=('_me', '_partner')
)
df_merged = df_merged.drop(columns = ['iid_me', 'pid', 'iid_partner'])

# Danh sách các sở thích gốc (không có hậu tố)
interests = [
    'sports', 'tvsports', 'exercise', 'dining', 'museums', 'art', 
    'hiking', 'gaming', 'clubbing', 'reading', 'tv', 'theater', 
    'movies', 'concerts', 'music', 'shopping', 'yoga'
]

# Tạo danh sách cột tương ứng cho me và partner
cols_me = [feat + '_me' for feat in interests]
cols_partner = [feat + '_partner' for feat in interests]



In [203]:
from scipy.stats import pearsonr

def calculate_pearson(row):
    # Lấy vector sở thích của me và partner, loại bỏ null nếu có
    me = row[cols_me].values
    partner = row[cols_partner].values
    
    # Pearson yêu cầu ít nhất 2 phần tử và không được là hằng số
    if pd.isna(me).any() or pd.isna(partner).any():
        return None
    
    corr, _ = pearsonr(me, partner)
    return corr

df_merged['new_int_corr'] = df_merged.apply(calculate_pearson, axis=1)
print(df_merged[['new_int_corr','int_corr']])

      new_int_corr  int_corr
0         0.270069      0.14
1         0.554515      0.54
2         0.313243      0.16
3         0.660628      0.61
4         0.339203      0.21
...            ...       ...
8363      0.483125      0.64
8364      0.670397      0.71
8365     -0.354714     -0.46
8366      0.661546      0.62
8367      0.136708      0.01

[8368 rows x 2 columns]
